In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
import torch_geometric.transforms as T
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:
# Define the GAT model
# this implementation is credit to pytorch_geometric examples
class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, data):
        h, edge_index = data.x, data.edge_index

        h = F.dropout(h, p=0.6, training=self.training)
        h = F.elu(self.conv1(h, edge_index))
        h = F.dropout(h, p=0.6, training=self.training)
        h = self.conv2(h, edge_index)

        return h

# Load the datasets
citeseer_dataset = Planetoid(root='/tmp/Citeseer', name='Citeseer', transform=T.NormalizeFeatures())
data = citeseer_dataset[0]
data = data.to(device)

Processing...
Done!


In [3]:
h_channels = 64
heads = 8
model = GAT(citeseer_dataset.num_features, h_channels, citeseer_dataset.num_classes, heads)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
def train(model, data, train_mask, labels):
    model.train()

    optimizer.zero_grad()
    logits = model(data.cuda())
    loss = F.cross_entropy(logits[train_mask], labels[train_mask])
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

In [4]:
train(model, data, data.train_mask, data.y)

1.7917002439498901

In [5]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data)
    pred = out.argmax(dim=1)

    acc = (pred[data.test_mask] == data.y[data.test_mask]).sum().item() / data.test_mask.sum().item()
    return acc

for epoch in range(0, 200):
    loss = train(model, data, data.train_mask, data.y)
    acc = test()
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 1.7794, Accuracy: 0.1820
Epoch: 001, Loss: 1.7644, Accuracy: 0.1820
Epoch: 002, Loss: 1.7376, Accuracy: 0.1840
Epoch: 003, Loss: 1.7457, Accuracy: 0.2540
Epoch: 004, Loss: 1.7349, Accuracy: 0.3280
Epoch: 005, Loss: 1.7157, Accuracy: 0.4520
Epoch: 006, Loss: 1.7124, Accuracy: 0.5940
Epoch: 007, Loss: 1.6801, Accuracy: 0.6330
Epoch: 008, Loss: 1.6742, Accuracy: 0.6280
Epoch: 009, Loss: 1.6161, Accuracy: 0.5840
Epoch: 010, Loss: 1.6119, Accuracy: 0.5720
Epoch: 011, Loss: 1.6123, Accuracy: 0.5600
Epoch: 012, Loss: 1.5869, Accuracy: 0.5630
Epoch: 013, Loss: 1.5779, Accuracy: 0.5650
Epoch: 014, Loss: 1.5499, Accuracy: 0.5970
Epoch: 015, Loss: 1.5110, Accuracy: 0.6260
Epoch: 016, Loss: 1.4625, Accuracy: 0.6710
Epoch: 017, Loss: 1.4778, Accuracy: 0.6950
Epoch: 018, Loss: 1.4200, Accuracy: 0.6800
Epoch: 019, Loss: 1.3763, Accuracy: 0.6830
Epoch: 020, Loss: 1.3807, Accuracy: 0.6890
Epoch: 021, Loss: 1.3643, Accuracy: 0.6920
Epoch: 022, Loss: 1.3708, Accuracy: 0.7060
Epoch: 023,

In [6]:
torch.save(model.state_dict(), 'citeseer_gat.pt')